# Definining Cohort

**The goal of this notebook is to identify patients with advanced melanoma who received first-line treatment with combination immunotherapy (e.g., nivolumab plus ipilimumab) or targeted therapy, specifically BRAF–MEK inhibitor combinations (e.g., dabrafenib plus trametinib, encorafenib plus binimetinib, or vemurafenib plus cobimetinib). Treatments were administered for metastatic or locoregional disease. Patients who received immunotherapy or targeted therapy in the adjuvant or neoadjuvant setting are excluded.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## 1. Identify patients receiving IO or TKI in adjuvant/neoadjuvant setting

In [3]:
therapy = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
(
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .PatientID.nunique()
)

9050

In [5]:
exclude_agents = [
    'Nivolumab',
    'Pembrolizumab',
    'Ipilimumab',
    'Nivolumab',
    'Dabrafenib',
    'Trametinib',
    'Binimetinib',
    'Encorafenib',
    'Vemurafenib'
]

pattern = "|".join(exclude_agents)

prior_io_tki_ids = (
    therapy
    .query('LineSetting in ["ADJUVANT", "NEOADJUVANT"]')
    .query('LineName.str.contains(@pattern, case=False, na=False)', engine="python")
    .PatientID
    .unique()
)

## 2. Combination immunotherapy 

In [6]:
therapy.query('LineNumber == 1').LineSetting.value_counts(dropna = False).head(20)

LineSetting
METASTATIC      8157
LOCOREGIONAL    1577
Name: count, dtype: int64

In [7]:
therapy.query('LineNumber == 1').IsMaintenanceTherapy.value_counts(dropna = False).head(20)

IsMaintenanceTherapy
False    9050
True      684
Name: count, dtype: int64

In [8]:
(
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .LineName.value_counts()
    .head(20)
)

LineName
Ipilimumab,Nivolumab                 2131
Pembrolizumab                        1576
Nivolumab                            1400
Ipilimumab                            819
Dabrafenib,Trametinib                 724
Clinical Study Drug                   628
Vemurafenib                           319
Binimetinib,Encorafenib               198
Nivolumab/Relatlimab-Rmbw             172
Temozolomide                          142
Cobimetinib,Vemurafenib                71
Talimogene Laherparepvec               67
Clinical Study Drug,Ipilimumab         61
Interferon Alfa-2B                     39
Dabrafenib                             33
Carboplatin,Paclitaxel                 33
Trametinib                             25
Ipilimumab,Pembrolizumab               21
Clinical Study Drug,Nivolumab          20
Clinical Study Drug,Pembrolizumab      19
Name: count, dtype: int64

In [9]:
ioio_df = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Ipilimumab,Nivolumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'ioio'))

In [10]:
ioio_df.sample(3)

,PatientID,LineName,StartDate
20119,F4D2539A99D3E,ioio,2021-03-26
20207,F25D11ACB2F79,ioio,2020-12-29
6522,FB1451BC9DC76,ioio,2022-03-04


In [11]:
row_ID(ioio_df)

(2131, 2131)

## 3. BRAF plus MEK inhibitor therapy

In [12]:
tki_df = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Dabrafenib,Trametinib" or LineName == "Binimetinib,Encorafenib" or LineName == "Cobimetinib,Vemurafenib"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'tki'))

In [13]:
tki_df.sample(3)

,PatientID,LineName,StartDate
5050,F7182D9B51CDD,tki,2016-02-09
23368,F4DDDC75463A9,tki,2019-01-15
5237,F3775652A630D,tki,2017-03-28


In [14]:
row_ID(tki_df)

(993, 993)

## 4. Combine dataframes and export to csv 

In [15]:
firstline_ioio_tki_index = pd.concat([ioio_df, tki_df], axis = 0)

In [16]:
row_ID(firstline_ioio_tki_index)

(3124, 3124)

In [17]:
# Remove patients with prior IO or TKI recepit in the adjuvant or neoadjuvant setting
firstline_ioio_tki_index = firstline_ioio_tki_index[~firstline_ioio_tki_index.PatientID.isin(prior_io_tki_ids)]

In [18]:
row_ID(firstline_ioio_tki_index)

(2771, 2771)

In [19]:
firstline_ioio_tki_index.sample(3)

,PatientID,LineName,StartDate
8818,F1C5A0B5297E8,ioio,2023-05-18
17507,F42B6FF8C91BD,tki,2022-05-25
6164,F08F5007EDBBD,tki,2014-11-19


In [20]:
firstline_ioio_tki_index.LineName.value_counts()

LineName
ioio    1923
tki      848
Name: count, dtype: int64

In [21]:
firstline_ioio_tki_index.to_csv('../outputs/ioio_tki_index.csv', index = False)